# 第 0 课：学习地图与实验环境

> **课程目标**：不是背算法，而是能够从数学目标推导更新公式、自己写训练循环、验证算法，再迁移到二维圆柱绕流与代理模型环境。

你已经有 Python / C、回归、随机森林、MLP 的初步知识；因此这门课采用 **理论 → 最小实现 → 可视化 → 单元测试 → 开放练习** 的形式。

## 学习路线

| 课 | 主题 | 代码里要会做什么 |
|---|---|---|
| 01 | MDP / Bellman | 手写价值迭代 |
| 02 | TD / Q-learning | 自己构造交互环境并学习策略 |
| 03 | Policy Gradient | 利用 JAX 自动微分优化随机策略 |
| 04 | Actor-Critic / GAE | 正确计算优势估计与时间截断 |
| 05 | PPO 连续控制 | 从零写一个 JAX PPO 训练器 |
| 06 | DQN / TD3 / SAC | 看懂关键损失与目标值的代码差异 |
| 07 | Model-based RL | 验证代理模型误差在闭环下的放大 |
| 08 | CFD 流动控制接口 | 明确观测、动作、奖励与物理验证 |
| 09 | 科研代码与复现 | 使用测试和实验记录约束模型结果 |

这里的 CFD 环境以**教学替代系统**开场，**不是**真实 Navier–Stokes 求解器；等你完成自己的 JAX CFD 求解器后再替换环境的动力学部分。

## 0.1 数学工具箱

一条核心链路：

$$
\text{状态 }s_t \xrightarrow{\text{策略 }\pi_\theta(a|s)} a_t
\xrightarrow{\text{动力学 }P(s'|s,a)}s_{t+1},r_t.
$$

本课程将反复使用：向量与矩阵、条件概率、期望、链式法则、梯度下降、时间序列折扣求和。训练前先看清**数组的轴和 shape**。

In [ ]:
import sys, numpy as np
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
# 一个 batch 包含 8 条轨迹，每条轨迹 16 步，每步 4 个观测量
observations = np.zeros((8, 16, 4))
actions = np.zeros((8, 16, 1))
print("obs shape:", observations.shape, "act shape:", actions.shape)
assert observations.shape[:-1] == actions.shape[:-1]

## 0.2 JAX：可微分计算，而不是更换 Python 语法

对于损失函数 $L(\theta)$，你既要计算数值 $L$，也要计算导数 $\partial L/\partial\theta$。
JAX 的 `grad` 提供梯度，`jit` 编译数值计算，`vmap` 批量向量化。随机数使用显式 key；后面的算法会实际使用它们。

In [ ]:
import jax
import jax.numpy as jnp

def f(theta):
    return jnp.sum((theta - jnp.array([1., -2.])) ** 2)

theta = jnp.array([0., 0.])
print("loss:", float(f(theta)), "gradient:", np.asarray(jax.grad(f)(theta)))
assert np.allclose(np.asarray(jax.grad(f)(theta)), [-2., 4.])
print("device:", jax.devices())

## 完成标准与练习

1. 修改数组 shape，观察广播规则何时导致静默错误。
2. 把 `f` 换成 $f(\theta)=\theta^T A\theta$，核对 $\nabla_\theta f=(A+A^T)\theta$。
3. 阅读：[JAX 快速入门](https://docs.jax.dev/en/latest/notebooks/quickstart.html)、[Gymnasium API](https://gymnasium.farama.org/api/env/)。